# Reddit Gender Classification & Masking Pipeline

This notebook contains the complete pipeline for analyzing and classifying gender based on Reddit posts. It handles data loading, duplicate cleaning, author-split management, noun & gender leakage masking (using spaCy), linguistic stylometric feature extraction, classic ML models, transformer fine-tuning, and qualitative analysis.

### Key Notebook Features:
- **Google Colab Integration**: Mounts Google Drive and installs/downloads raw files automatically if run in Colab.
- **Persistent Caching**: Saves pre-processed (masked) datasets to disk/drive to skip time-consuming NLP masking steps.
- **Model Checkpointing**: Saves and loads classic ML pipelines and Transformer weights so training doesn't have to be repeated.
- **Granular Control**: Cells are modularized to run specific components individually.
- **Beautiful Outputs**: Uses styled pandas tables and clean summaries for presentation.

## Step 1: Environment Setup & Drive Mounting

In [ ]:
import os
import sys

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Define base paths (change this path according to your Google Drive layout)
    DRIVE_DIR = '/content/drive/MyDrive/NLP-project'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    os.makedirs(os.path.join(DRIVE_DIR, 'data'), exist_ok=True)
    os.makedirs(os.path.join(DRIVE_DIR, 'models'), exist_ok=True)
    os.makedirs(os.path.join(DRIVE_DIR, 'results'), exist_ok=True)
    
    DATA_DIR = os.path.join(DRIVE_DIR, 'data')
    MODELS_DIR = os.path.join(DRIVE_DIR, 'models')
    RESULTS_DIR = os.path.join(DRIVE_DIR, 'results')
    
    # Install gdown and spacy dependencies
    !pip install -q gdown spacy transformers datasets joblib
    !python -m spacy download en_core_web_sm
else:
    print("Running locally.")
    DATA_DIR = 'data'
    MODELS_DIR = 'models'
    RESULTS_DIR = 'results'
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(MODELS_DIR, exist_ok=True)
    os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Models directory: {MODELS_DIR}")
print(f"Results directory: {RESULTS_DIR}")

## Step 2: Download/Load Raw Dataset

In [ ]:
import gdown

raw_csv_path = os.path.join(DATA_DIR, 'gender.csv')
url = 'https://drive.google.com/uc?id=14poUE5BV9EVm9ofUnp79oHd_AC6SeLpw'

if not os.path.exists(raw_csv_path):
    print(f"Downloading gender.csv into {raw_csv_path}...")
    gdown.download(url, raw_csv_path, quiet=False)
else:
    print(f"gender.csv already exists at {raw_csv_path}")

## Step 3: Data Loading, Splitting, and Cleaning
We load the data, split by author ID (to prevent authorship-based data leakage), and remove test records that are near-duplicates of the training set.

In [ ]:
import pandas as pd
# Import utilities from the source files
from prepare_data import load_data, split_by_author, remove_near_duplicates
from config import DATA_FILE

# Temporarily override DATA_FILE to point to our chosen directory
import config
config.DATA_FILE = raw_csv_path

print("Loading dataset...")
df = load_data()

print("\nSplitting data by author (Train/Test)...")
train_df, test_df = split_by_author(df)

print("\nRemoving near-duplicates between train and test sets...")
test_df = remove_near_duplicates(train_df, test_df)

## Step 4: Masking (Leakage & Noun Removal)

Using spaCy POS tagging and lemmatization, we mask gendered terms and nouns to analyze stylometric signal degradation. 

**Persistence Logic**: If pre-processed datasets already exist under `DATA_DIR`, we load them directly instead of running the slow spaCy process again.

In [ ]:
from masking import apply_masking

train_masked_path = os.path.join(DATA_DIR, 'train_masked.csv')
test_masked_path = os.path.join(DATA_DIR, 'test_masked.csv')

# Check if masked datasets already exist to skip long preprocessing
load_existing_masked = True  # Set to False to force re-running the masking pipeline

if load_existing_masked and os.path.exists(train_masked_path) and os.path.exists(test_masked_path):
    print("Loading saved masked datasets persistently...")
    train_df = pd.read_csv(train_masked_path)
    test_df = pd.read_csv(test_masked_path)
    print(f"Loaded train set ({len(train_df)} rows) and test set ({len(test_df)} rows)")
else:
    print("Masked datasets not found (or force re-run enabled). Applying spaCy masking (this may take a while)...")
    train_df, test_df = apply_masking(train_df, test_df)
    
    # Save persistently
    train_df.to_csv(train_masked_path, index=False)
    test_df.to_csv(test_masked_path, index=False)
    print(f"Saved masked datasets persistently to:\n  - {train_masked_path}\n  - {test_masked_path}")

## Step 5: Traditional ML Experiments (TF-IDF)

We train classic classifiers (MultinomialNB, Logistic Regression, Linear SVM) across three configurations: `original`, `leakage_masked`, and `noun_masked`.

**Persistence Logic**: If pre-trained models exist, we load them. Otherwise, we train them and save them.

In [ ]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

models = {
    "MultinomialNB": MultinomialNB(),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "LinearSVM": LinearSVC()
}

settings = {
    "original": "text",
    "leakage_masked": "text_leakage_masked",
    "noun_masked": "text_noun_masked"
}

results = []
trained_pipelines = {}

# Set to True to load pre-trained pipelines if they exist
load_saved_classic_models = True

for model_name, model in models.items():
    trained_pipelines[model_name] = {}
    for setting_name, column in settings.items():
        model_filename = os.path.join(MODELS_DIR, f"{model_name}_{setting_name}.joblib")
        
        if load_saved_classic_models and os.path.exists(model_filename):
            print(f"Loading pre-trained {model_name} ({setting_name}) model...")
            pipe = joblib.load(model_filename)
        else:
            print(f"Training {model_name} on {setting_name} content...")
            pipe = Pipeline([
                ("tfidf", TfidfVectorizer(
                    ngram_range=(1, 2),
                    min_df=5,
                    max_df=0.9,
                    stop_words="english"
                )),
                ("clf", model)
            ])
            pipe.fit(train_df[column].astype(str), train_df["label"])
            joblib.dump(pipe, model_filename)
            
        trained_pipelines[model_name][setting_name] = pipe
        
        # Evaluate
        preds = pipe.predict(test_df[column].astype(str))
        acc = accuracy_score(test_df["label"], preds)
        f1 = f1_score(test_df["label"], preds, average="macro")
        
        results.append({
            "model": model_name,
            "setting": setting_name,
            "features": "tfidf",
            "accuracy": acc,
            "macro_f1": f1
        })

results_df = pd.DataFrame(results)

print("\n" + "=" * 60)
print("CLASSIC ML EXPERIMENT RESULTS (TF-IDF)")
print("=" * 60)
# Display results in a beautifully styled gradient table
pivot_f1 = results_df.pivot(index="model", columns="setting", values="macro_f1")
display(pivot_f1.style.background_gradient(cmap='Blues').format("{:.4f}").set_caption("Macro F1-Score"))

pivot_acc = results_df.pivot(index="model", columns="setting", values="accuracy")
display(pivot_acc.style.background_gradient(cmap='Greens').format("{:.4f}").set_caption("Accuracy"))

## Step 6: Stylometric Experiments (spaCy features)

Extract and evaluate purely structural linguistic features (POS tag frequencies, punctuation usage, sentence counts) with Logistic Regression and Random Forest models.

In [ ]:
from spacy_features import extract_spacy_features
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

train_spacy_path = os.path.join(DATA_DIR, 'train_spacy_features.csv')
test_spacy_path = os.path.join(DATA_DIR, 'test_spacy_features.csv')

load_saved_spacy_features = True

if load_saved_spacy_features and os.path.exists(train_spacy_path) and os.path.exists(test_spacy_path):
    print("Loading saved spaCy features...")
    train_spacy_features = pd.read_csv(train_spacy_path)
    test_spacy_features = pd.read_csv(test_spacy_path)
else:
    print("Extracting spaCy linguistic/stylometric features...")
    train_spacy_features = extract_spacy_features(train_df["text"].tolist(), batch_size=50, chunk_size=2000)
    test_spacy_features = extract_spacy_features(test_df["text"].tolist(), batch_size=50, chunk_size=2000)
    
    # Save for persistence
    train_spacy_features.to_csv(train_spacy_path, index=False)
    test_spacy_features.to_csv(test_spacy_path, index=False)
    print("spaCy features extracted and cached successfully.")

print(f"Features extracted: {train_spacy_features.shape[1]} columns.")

# Scale and train models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_spacy_features)
X_test_scaled = scaler.transform(test_spacy_features)

spacy_models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42)
}

spacy_results = []

for model_name, model in spacy_models.items():
    model_filename = os.path.join(MODELS_DIR, f"{model_name}_spacy_stylometric.joblib")
    
    if load_saved_classic_models and os.path.exists(model_filename):
        print(f"Loading pre-trained {model_name} (stylometric) model...")
        model = joblib.load(model_filename)
    else:
        print(f"Training {model_name} on stylometric features...")
        model.fit(X_train_scaled, train_df["label"])
        joblib.dump(model, model_filename)
        
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(test_df["label"], preds)
    f1 = f1_score(test_df["label"], preds, average="macro")
    
    spacy_results.append({
        "model": model_name,
        "setting": "spacy_only",
        "features": "spacy_linguistic",
        "accuracy": acc,
        "macro_f1": f1
    })

spacy_results_df = pd.DataFrame(spacy_results)
print("\nStylometric Model Performance (Pure style, no word tokens):")
display(spacy_results_df.style.background_gradient(cmap='Purples', subset=['accuracy', 'macro_f1']).format({
    'accuracy': '{:.4f}',
    'macro_f1': '{:.4f}'
}))

## Step 7: Transformer Experiments (Fine-tuning DistilBERT)

We load/fine-tune DistilBERT using Hugging Face. The cell will process a specific text columns sequence: `text`, `text_leakage_masked`, and `text_noun_masked`.

**Persistence Logic**: Models are saved to `MODELS_DIR`. If directories already contain a trained Hugging Face model checkpoint, the pipeline loads them directly to compute metrics without triggering another epoch of fine-tuning.

In [ ]:
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

model_name = "distilbert-base-uncased"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

transformer_results = []
load_saved_transformer = True  # Load previously trained checkpoint if available

for setting_name, text_col in [("original", "text"), ("leakage_masked", "text_leakage_masked"), ("noun_masked", "text_noun_masked")]:
    checkpoint_dir = os.path.join(MODELS_DIR, f"distilbert_{setting_name}")
    print(f"\n--- Fine-tuning/Loading DistilBERT for setting: {setting_name} ({text_col}) ---")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    unique_labels = train_df["label"].unique()
    label2id = {str(label): i for i, label in enumerate(sorted(unique_labels))}
    id2label = {i: str(label) for label, i in label2id.items()}
    num_labels = len(unique_labels)
    
    train_dataset = Dataset.from_pandas(train_df[[text_col, "label"]])
    test_dataset = Dataset.from_pandas(test_df[[text_col, "label"]])
    
    def tokenize_function(examples):
        tokens = tokenizer(examples[text_col].astype(str).tolist(), padding="max_length", truncation=True, max_length=128)
        tokens["labels"] = [label2id[str(label)] for label in examples["label"]]
        return tokens
        
    train_dataset = train_dataset.map(tokenize_function, batched=True)
    test_dataset = test_dataset.map(tokenize_function, batched=True)
    
    # Check if a model checkpoint already exists
    if load_saved_transformer and (os.path.exists(os.path.join(checkpoint_dir, "pytorch_model.bin")) or os.path.exists(os.path.join(checkpoint_dir, "model.safetensors"))):
        print(f"Found existing checkpoint in {checkpoint_dir}. Loading weights directly...")
        model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
        training_args = TrainingArguments(
            output_dir=checkpoint_dir,
            per_device_eval_batch_size=16,
            report_to="none"
        )
        trainer = Trainer(
            model=model,
            args=training_args,
            eval_dataset=test_dataset,
            compute_metrics=lambda eval_pred: {
                "accuracy": accuracy_score(eval_pred.label_ids, np.argmax(eval_pred.predictions, axis=-1)),
                "macro_f1": f1_score(eval_pred.label_ids, np.argmax(eval_pred.predictions, axis=-1), average="macro")
            }
        )
    else:
        print(f"Training DistilBERT from scratch (epochs=3)...")
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=num_labels, label2id=label2id, id2label=id2label
        )
        
        training_args = TrainingArguments(
            output_dir=checkpoint_dir,
            evaluation_strategy="epoch",
            save_strategy="epoch",
            learning_rate=2e-5,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            num_train_epochs=3,
            weight_decay=0.01,
            logging_steps=50,
            load_best_model_at_end=True,
            report_to="none"
        )
        
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            compute_metrics=lambda eval_pred: {
                "accuracy": accuracy_score(eval_pred.label_ids, np.argmax(eval_pred.predictions, axis=-1)),
                "macro_f1": f1_score(eval_pred.label_ids, np.argmax(eval_pred.predictions, axis=-1), average="macro")
            }
        )
        
        trainer.train()
        # Save final best model locally
        trainer.save_model(checkpoint_dir)
        tokenizer.save_pretrained(checkpoint_dir)
        print(f"Model checkpoint successfully saved to: {checkpoint_dir}")
        
    eval_results = trainer.evaluate()
    acc = eval_results["eval_accuracy"]
    f1 = eval_results["eval_macro_f1"]
    
    transformer_results.append({
        "model": "distilbert-base-uncased",
        "setting": setting_name,
        "features": "transformer",
        "accuracy": acc,
        "macro_f1": f1
    })

transformer_df = pd.DataFrame(transformer_results)
# Combine all results and save them
combined_results_df = pd.concat([results_df, transformer_df], ignore_index=True)
combined_results_df.to_csv(os.path.join(RESULTS_DIR, 'results.csv'), index=False)
print(f"\nSaved all experimental results to {os.path.join(RESULTS_DIR, 'results.csv')}")

print("\n--- Transformer Evaluation Performance ---")
display(transformer_df.style.background_gradient(cmap='Oranges', subset=['accuracy', 'macro_f1']).format({
    'accuracy': '{:.4f}',
    'macro_f1': '{:.4f}'
}))

## Step 8: Qualitative Analysis & Feature Importance Comparison

We perform qualitative checks on predictions from original and masked models, identifying instances where predictions flipped under masking configurations, and saving feature correlation rankings.

In [ ]:
from qualitative import create_masked_analysis_csv

# Re-instantiate pipelines from dictionary for qualitative analysis utility
orig_pipe = trained_pipelines["LogisticRegression"]["original"]
leakage_pipe = trained_pipelines["LogisticRegression"]["leakage_masked"]
noun_pipe = trained_pipelines["LogisticRegression"]["noun_masked"]

# Compute qualitative comparisons
print("Comparing original LogisticRegression and leakage masked LogisticRegression...")
leakage_analysis_df = create_masked_analysis_csv(
    test_df,
    orig_pipe,
    leakage_pipe,
    masked_col="text_leakage_masked",
    output_name="leakage"
)

print("\nComparing original LogisticRegression and noun masked LogisticRegression...")
noun_analysis_df = create_masked_analysis_csv(
    test_df,
    orig_pipe,
    noun_pipe,
    masked_col="text_noun_masked",
    output_name="noun"
)

# Show sample changed cases
flipped_cases = leakage_analysis_df[leakage_analysis_df['prediction_changed']].head(5)
print("\nTop Prediction-Changed Examples in Leakage-Masked Test Set:")
display(flipped_cases[['original_text', 'masked_text', 'true_label', 'original_prediction', 'masked_prediction']])